# Secure Blockchain Programming in Python
**Google Colab Edition**

This notebook walks through two exercises:

1. **Build a blockchain from genesis**, with proof-of-work and tamper detection.
2. **Build wallets and signed transactions**, with replay-attack and double-spend prevention.

Every code block runs top-to-bottom. The focus is on **security** — each section ends with a checklist of bugs to look for during code review.

> **Note on language choice.** Production blockchains are rarely written in Python — no secure memory zeroing, slow crypto, GC unpredictability. We use Python here because it lowers the barrier to focusing on *concepts* and *vulnerabilities*. Every security principle here transfers directly to Rust / Go / C++.


## Setup — install & import dependencies

Run this once at the top of the notebook.

In [ ]:
# Install cryptography library for ECDSA on secp256k1.
# 'ecdsa' is pure Python, which is perfect for a teaching environment.
# For production you would use 'coincurve' (libsecp256k1 bindings) — 100x faster.
!pip install -q ecdsa==0.19.0
print("dependencies installed")

In [ ]:
import hashlib
import json
import time
import secrets                         # cryptographically secure RNG - NEVER use 'random' for keys
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Dict, Tuple
from ecdsa import SigningKey, VerifyingKey, SECP256k1, BadSignatureError

---
# Exercise 1: Blockchain from Genesis

## Learning Objectives
- Build a tamper-evident, hash-linked chain of blocks.
- Implement proof-of-work mining with adjustable difficulty.
- Validate the entire chain and detect tampering.
- Understand *why* each field in a block exists.

## Concepts (recap)
1. **Hash functions** — pre-image resistance, collision resistance, avalanche.
2. **Hash-linked lists** — why `prev_hash` makes the structure append-only *in practice*.
3. **Proof-of-work** — why we require `hash` to start with N zeros, and why difficulty matters.
4. **Genesis** — the one block whose `prev_hash` is hardcoded, and why determinism matters.
5. **Chain validation** — what `is_valid()` must check (and what students usually forget).


### The `Block` class

Read the `compute_hash` method carefully — **every mutable field must be included, in a deterministic order, with a fixed byte encoding**. Two nodes must agree byte-for-byte, or the chain forks.

In [ ]:
MAX_DATA_BYTES = 64 * 1024     # hard cap on block data - prevents memory DoS
MAX_U64        = 2**64 - 1     # used to bound nonce and timestamp

@dataclass
class Block:
    index: int
    timestamp: int          # unix seconds, unsigned
    data: str               # plain string for week 1; becomes List[Transaction] in week 2
    prev_hash: str          # hex
    nonce: int = 0
    hash: str = ""

    def compute_hash(self) -> str:
        """Deterministic SHA-256 over the header fields.

        SECURITY: byte order is fixed to big-endian, strings are UTF-8 encoded,
        and the order of updates is part of the protocol. Changing any of this
        silently forks the chain.
        """
        # Bound-check to force overflow into an explicit error rather than
        # silently producing a different hash on a 32-bit machine etc.
        if not (0 <= self.index <= MAX_U64):       raise OverflowError("index")
        if not (0 <= self.timestamp <= MAX_U64):   raise OverflowError("timestamp")
        if not (0 <= self.nonce <= MAX_U64):       raise OverflowError("nonce")
        if len(self.data.encode("utf-8")) > MAX_DATA_BYTES:
            raise ValueError("data too large")

        h = hashlib.sha256()
        h.update(self.index.to_bytes(8, "big"))
        h.update(self.timestamp.to_bytes(8, "big"))
        h.update(self.data.encode("utf-8"))
        h.update(self.prev_hash.encode("utf-8"))
        h.update(self.nonce.to_bytes(8, "big"))
        return h.hexdigest()

    def mine(self, difficulty: int) -> None:
        """Find a nonce such that hash starts with `difficulty` zeros."""
        target = "0" * difficulty
        while True:
            self.hash = self.compute_hash()
            if self.hash.startswith(target):
                return
            if self.nonce >= MAX_U64:
                raise OverflowError("nonce exhausted")
            self.nonce += 1

### The `Blockchain` class

Note the **hardcoded genesis timestamp of 0**. If you used `time.time()` here, every student's genesis block would hash differently, and no two chains could ever agree. That is a real consensus bug.

In [ ]:
class ChainError(Exception): pass

class Blockchain:
    def __init__(self, difficulty: int = 4):
        self.difficulty = difficulty
        genesis = Block(
            index=0,
            timestamp=0,                # HARDCODED - genesis must be deterministic
            data="GENESIS",
            prev_hash="0" * 64,
        )
        genesis.mine(difficulty)
        self.chain: List[Block] = [genesis]

    def add(self, data: str) -> Block:
        prev = self.chain[-1]
        block = Block(
            index=prev.index + 1,
            timestamp=int(time.time()),
            data=data,
            prev_hash=prev.hash,
        )
        block.mine(self.difficulty)
        self.chain.append(block)
        return block

    def is_valid(self) -> Tuple[bool, str]:
        """Return (ok, reason). Reason is empty on success."""
        target = "0" * self.difficulty
        for i in range(1, len(self.chain)):
            cur, prev = self.chain[i], self.chain[i - 1]
            if cur.compute_hash() != cur.hash:
                return False, f"block {i}: hash mismatch (tampered)"
            if cur.prev_hash != prev.hash:
                return False, f"block {i}: broken link to previous block"
            if not cur.hash.startswith(target):
                return False, f"block {i}: insufficient proof-of-work"
            if cur.timestamp < prev.timestamp:
                return False, f"block {i}: non-monotonic timestamp"
            if cur.index != prev.index + 1:
                return False, f"block {i}: index out of order"
        return True, ""

### Smoke test — mine three blocks

In [ ]:
chain = Blockchain(difficulty=4)
for payload in ["Alice pays Bob 10", "Bob pays Carol 3", "Carol pays Alice 1"]:
    blk = chain.add(payload)
    print(f"mined block {blk.index}  nonce={blk.nonce:>6}  hash={blk.hash[:16]}...")

ok, why = chain.is_valid()
print("\nchain valid?", ok, why)

### Tamper-detection test

Three attacks to try:

1. **Naive tamper** — change `data` and leave `hash` alone. Validation catches the `hash` mismatch.
2. **Sneaky tamper** — change `data` **and** recompute `hash`. Validation still fails, because the recomputed hash no longer satisfies the proof-of-work target.
3. **Full rewrite** — change `data`, re-mine the block. Validation still fails because the next block's `prev_hash` is now stale.



In [ ]:
# Attack 1 - naive: tamper without rehash
bad = Blockchain(difficulty=4)
bad.add("real tx 1")
bad.add("real tx 2")
bad.chain[1].data = "EVIL: send 1000 to attacker"
print("naive tamper ->", bad.is_valid())

# Attack 2 - sneaky: tamper AND rehash (but skip re-mining)
bad2 = Blockchain(difficulty=4)
bad2.add("real tx 1")
bad2.add("real tx 2")
bad2.chain[1].data = "EVIL: send 1000 to attacker"
bad2.chain[1].hash = bad2.chain[1].compute_hash()    # recompute
print("sneaky tamper ->", bad2.is_valid())

# Attack 3 - full rewrite: re-mine block 1, but block 2 still links to the OLD hash
bad3 = Blockchain(difficulty=4)
bad3.add("real tx 1")
bad3.add("real tx 2")
bad3.chain[1].data = "EVIL: send 1000 to attacker"
bad3.chain[1].nonce = 0
bad3.chain[1].mine(bad3.difficulty)
print("rewrite without re-mining tail ->", bad3.is_valid())

### Student Tasks — Week 1

#### Tier 1 — Must complete
1. Run the chain at `difficulty=5` and `difficulty=6`. Time how long mining takes. Plot or tabulate.
2. Write a test that flips a single bit in `chain[1].data` and confirms `is_valid()` fails.
3. Write a test that tampers with `chain[1].data` **and** re-mines that block — and explain why validation *still* fails (hint: what about block 2?).

#### Tier 2 — Should complete
4. Implement `replace_chain(other: List[Block])` that replaces the local chain only if the other is (a) longer and (b) valid. This is the *longest-valid-chain* rule.
5. Serialise the chain to JSON, deserialise, and confirm it still validates. (Byte-for-byte agreement!)
6. Add a `max_future_skew` check — reject blocks whose timestamp is more than 2 hours in the future. Why does this matter?

#### Tier 3 — Stretch
7. Make difficulty adjust every 10 blocks to target ~10 s/block.
8. Add a Merkle root over `data` items (prepares for Week 2).
9. Add a property test: for any random sequence of `add()` calls, `is_valid()` is always `Ok`.


### Starter stub for Task 4 (longest-valid-chain)

In [ ]:
def replace_chain(self: Blockchain, other: List[Block]) -> bool:
    """Replace self.chain with `other` if it is longer AND valid."""
    if len(other) <= len(self.chain):
        return False
    # Temporarily swap and validate, so we don't corrupt on failure.
    backup = self.chain
    self.chain = other
    ok, _ = self.is_valid()
    if not ok:
        self.chain = backup
        return False
    return True

Blockchain.replace_chain = replace_chain  # monkey-patch for the exercise

# Students: write tests that prove this function is safe against (a) short chains,
# (b) invalid chains, (c) chains with a different genesis block.

---
# Exercise 2: Wallet, Sign, Send, Receive

## Learning Objectives
- Generate ECDSA keypairs on secp256k1 (the curve used by Bitcoin and Ethereum).
- Derive a wallet **address** from a public key.
- Build, sign, verify, and apply transactions.
- Maintain state (balances + nonces) and prevent replay / double-spend.

## Concepts (recap)
1. **Asymmetric signatures** — private key signs, public key verifies. Address = hash of public key.
2. **Nonces** — a per-sender counter. Without it, Alice's "send 10 to Bob" can be replayed forever.
3. **Signature malleability** — ECDSA has a quirk where `(r, s)` and `(r, -s mod n)` are both valid. A well-maintained library like `ecdsa` handles this; a hand-rolled implementation probably doesn't.
4. **Mempool & inclusion** — transactions live in a pool, miners include them in blocks.
5. **State vs history** — the chain is history; balances are state derived from replaying history.


### The `Wallet` class

Two production caveats we **cannot** fully address in Python:

- **Zeroizing key material.** CPython keeps byte strings live until GC; there is no safe `zeroize()` equivalent. In Rust you would use the `zeroize` crate. Here we just keep the attribute underscored and document the risk.
- **Key storage.** Writing `signing_key` to a plain file is obviously wrong. In a real wallet you'd use the OS keychain, an HSM, or a hardware wallet.

In [ ]:
ADDR_LEN = 40   # hex chars -> 20 bytes, same shape as an Ethereum address

class Wallet:
    def __init__(self):
        # ecdsa uses os.urandom under the hood, which is cryptographically secure.
        # Explicitly pass `secrets.token_bytes` to make the intent obvious.
        self._signing_key: SigningKey = SigningKey.generate(
            curve=SECP256k1, entropy=secrets.token_bytes
        )
        self.verifying_key: VerifyingKey = self._signing_key.verifying_key

    # Public key as compressed SEC1 (33 bytes) - canonical form, smaller, deterministic.
    def public_key_bytes(self) -> bytes:
        return self.verifying_key.to_string("compressed")

    def address(self) -> str:
        return hashlib.sha256(self.public_key_bytes()).hexdigest()[:ADDR_LEN]

    def sign(self, msg: bytes) -> bytes:
        # Deterministic signatures (RFC 6979) would be better; the default
        # `sign` uses random k. For teaching purposes this is fine - but note:
        # NEVER reuse k across two signatures with the same key (this leaks
        # the private key - see the 2010 Sony PS3 breach).
        return self._signing_key.sign(msg, hashfunc=hashlib.sha256)

    def __repr__(self):
        return f"Wallet(addr={self.address()})"

### The `Transaction` class

Three things to notice:

1. `signing_hash` covers **everything except the signature itself**. Including the signature would be a chicken-and-egg: the hash would depend on the thing you're trying to produce.
2. `verify` **re-derives the address** from the attached public key and checks it matches `from_addr`. Without this, anyone could claim "I am Alice" while presenting their own public key.
3. `amount == 0` and `from_addr == to_addr` are rejected — both are classic inflation / duplication bugs.

In [ ]:
class TxError(Exception): pass

MAX_AMOUNT = 2**64 - 1

@dataclass
class Transaction:
    from_addr: str
    to_addr:   str
    amount:    int
    nonce:     int
    public_key: str = ""   # hex, compressed SEC1
    signature:  str = ""   # hex

    # ---- canonical byte layout for signing ---------------------------------
    def signing_hash(self) -> bytes:
        if not (0 < self.amount <= MAX_AMOUNT):
            raise TxError("amount out of range")
        if not (0 <= self.nonce <= MAX_U64):
            raise TxError("nonce out of range")
        if self.from_addr == self.to_addr:
            raise TxError("self-transfer not allowed")
        h = hashlib.sha256()
        h.update(self.from_addr.encode("utf-8"))
        h.update(self.to_addr.encode("utf-8"))
        h.update(self.amount.to_bytes(8, "big"))
        h.update(self.nonce.to_bytes(8, "big"))
        return h.digest()

    # ---- signing ----------------------------------------------------------
    def sign_with(self, wallet: "Wallet") -> None:
        if wallet.address() != self.from_addr:
            raise TxError("wallet does not own from_addr")
        self.public_key = wallet.public_key_bytes().hex()
        self.signature  = wallet.sign(self.signing_hash()).hex()

    # ---- verification -----------------------------------------------------
    def verify(self) -> bool:
        if not self.public_key or not self.signature:
            raise TxError("unsigned transaction")

        pk_bytes = bytes.fromhex(self.public_key)

        # Re-derive the address from the pubkey and compare.
        # Without this, anyone could attach their own pubkey + sig and
        # claim to be `from_addr`.
        derived = hashlib.sha256(pk_bytes).hexdigest()[:ADDR_LEN]
        if derived != self.from_addr:
            raise TxError("public key does not match from_addr")

        vk = VerifyingKey.from_string(pk_bytes, curve=SECP256k1)
        try:
            vk.verify(bytes.fromhex(self.signature),
                      self.signing_hash(),
                      hashfunc=hashlib.sha256)
            return True
        except BadSignatureError:
            return False

### The `State` class (balances + nonces)

This is the "ledger" derived by replaying all transactions. A transaction is **valid** if and only if:

- its signature verifies,
- its nonce equals `state.nonces[sender]` (no gaps, no reuse),
- `state.balances[sender] >= amount`.

The check order matters: verify signature -> check nonce -> check balance -> apply atomically. If any check fails the state is not mutated.

In [ ]:
class State:
    def __init__(self, genesis_balances: Optional[Dict[str, int]] = None):
        self.balances: Dict[str, int] = dict(genesis_balances or {})
        self.nonces:   Dict[str, int] = {}    # next-expected nonce per sender

    def apply(self, tx: Transaction) -> None:
        # 1. cryptographic validity
        if not tx.verify():
            raise TxError("invalid signature")

        # 2. nonce check (replay / double-spend protection)
        expected = self.nonces.get(tx.from_addr, 0)
        if tx.nonce != expected:
            raise TxError(f"bad nonce (expected {expected}, got {tx.nonce})")

        # 3. balance check - BEFORE any mutation
        bal = self.balances.get(tx.from_addr, 0)
        if bal < tx.amount:
            raise TxError("insufficient balance")

        # 4. apply atomically (Python is single-threaded here - no locking
        #    needed, but in a real node this whole block is a critical section)
        self.balances[tx.from_addr] = bal - tx.amount
        self.balances[tx.to_addr]   = self.balances.get(tx.to_addr, 0) + tx.amount
        self.nonces[tx.from_addr]   = expected + 1

    def snapshot(self) -> Dict[str, int]:
        return dict(self.balances)

### End-to-end demo — Alice sends 10 to Bob

In [ ]:
alice = Wallet()
bob   = Wallet()
print("Alice:", alice.address())
print("Bob  :", bob.address())

# Give Alice some initial coins via a genesis allocation.
state = State(genesis_balances={alice.address(): 100})

tx = Transaction(from_addr=alice.address(),
                 to_addr=bob.address(),
                 amount=10,
                 nonce=0)
tx.sign_with(alice)

print("signature valid?", tx.verify())

state.apply(tx)
print("balances after:", state.snapshot())

### Attack demo 1 — tamper after signing

Classic "man in the middle edits the amount". The signature covers the amount, so verification fails.

In [ ]:
tx = Transaction(alice.address(), bob.address(), amount=10, nonce=1)
tx.sign_with(alice)
tx.amount = 10_000     # attacker inflates the amount after signing
try:
    # We expect verify() to return False OR apply() to raise.
    ok = tx.verify()
    if ok:
        state.apply(tx)
        print("FAIL: attack succeeded (BUG)")
    else:
        print("blocked: signature does not cover tampered amount")
except TxError as e:
    print("blocked:", e)

### Attack demo 2 — replay attack, prevented by the nonce

The attacker captures Alice's "send 10 to Bob" transaction from the network and re-submits it repeatedly. Without nonces, each resubmission would debit Alice another 10 coins. With nonces, only one can ever succeed.

In [ ]:
alice2 = Wallet()
bob2   = Wallet()
s = State(genesis_balances={alice2.address(): 100})

tx = Transaction(alice2.address(), bob2.address(), 10, nonce=0)
tx.sign_with(alice2)

s.apply(tx)                  # first submission - succeeds
try:
    s.apply(tx)              # second submission - must fail
    print("FAIL: replay succeeded (BUG)")
except TxError as e:
    print("blocked:", e)

print("Alice balance:", s.balances[alice2.address()])  # should still be 90

### Attack demo 3 — double-spend, prevented by the nonce

Alice tries to create **two different transactions with the same nonce** — say, sending 10 to Bob AND 10 to Carol at nonce=0. Only one can be included.

In [ ]:
alice3 = Wallet()
bob3   = Wallet()
carol3 = Wallet()
s = State(genesis_balances={alice3.address(): 15})

tx1 = Transaction(alice3.address(), bob3.address(),   10, nonce=0)
tx2 = Transaction(alice3.address(), carol3.address(), 10, nonce=0)    # same nonce!
tx1.sign_with(alice3)
tx2.sign_with(alice3)

s.apply(tx1)                 # first one wins
try:
    s.apply(tx2)
    print("FAIL: double-spend succeeded (BUG)")
except TxError as e:
    print("blocked:", e)

# Note: Alice had 15 coins. She tried to spend 20 total. Without the nonce
# check, her second transaction would at least be rejected by the balance
# check. But if the debit happened BEFORE the balance check, state would
# be corrupted. Nonce + balance check together close the hole.

### Integrating transactions into the blockchain

Replaces `data: str` with `data: List[Transaction]`. The hash input now includes a deterministic serialisation of every transaction, and the block is valid only if every transaction verifies.

In [ ]:
def tx_to_dict(tx: Transaction) -> dict:
    # Fixed key order for deterministic JSON - dict insertion order is preserved in 3.7+.
    return {
        "from_addr":  tx.from_addr,
        "to_addr":    tx.to_addr,
        "amount":     tx.amount,
        "nonce":      tx.nonce,
        "public_key": tx.public_key,
        "signature":  tx.signature,
    }

def tx_from_dict(d: dict) -> Transaction:
    return Transaction(**d)

@dataclass
class TxBlock:
    index: int
    timestamp: int
    txs: List[Transaction]
    prev_hash: str
    nonce: int = 0
    hash: str = ""

    def compute_hash(self) -> str:
        if not (0 <= self.nonce <= MAX_U64): raise OverflowError("nonce")
        payload = json.dumps([tx_to_dict(t) for t in self.txs],
                             sort_keys=False, separators=(",", ":")).encode("utf-8")
        if len(payload) > 1 << 20:            # 1 MiB cap per block
            raise ValueError("block too large")
        h = hashlib.sha256()
        h.update(self.index.to_bytes(8, "big"))
        h.update(self.timestamp.to_bytes(8, "big"))
        h.update(payload)
        h.update(self.prev_hash.encode("utf-8"))
        h.update(self.nonce.to_bytes(8, "big"))
        return h.hexdigest()

    def mine(self, difficulty: int) -> None:
        target = "0" * difficulty
        while True:
            self.hash = self.compute_hash()
            if self.hash.startswith(target): return
            if self.nonce >= MAX_U64: raise OverflowError("nonce exhausted")
            self.nonce += 1


class TxBlockchain:
    def __init__(self, difficulty: int = 3, genesis_balances: Optional[Dict[str,int]] = None):
        self.difficulty = difficulty
        self.state = State(genesis_balances)
        g = TxBlock(index=0, timestamp=0, txs=[], prev_hash="0"*64)
        g.mine(difficulty)
        self.chain: List[TxBlock] = [g]
        self.mempool: List[Transaction] = []

    def submit(self, tx: Transaction) -> None:
        if not tx.verify():
            raise TxError("invalid signature; not added to mempool")
        self.mempool.append(tx)

    def mine_block(self) -> TxBlock:
        """Include as many valid mempool txs as possible, in nonce order per sender."""
        accepted: List[Transaction] = []
        # simulate state application to filter
        sim = State(self.state.balances.copy())
        sim.nonces = self.state.nonces.copy()
        for tx in self.mempool:
            try:
                sim.apply(tx)
                accepted.append(tx)
            except TxError:
                continue    # drop invalid/out-of-order tx for this round

        prev = self.chain[-1]
        blk = TxBlock(index=prev.index + 1,
                      timestamp=int(time.time()),
                      txs=accepted,
                      prev_hash=prev.hash)
        blk.mine(self.difficulty)

        # Commit to real state only after successful mining.
        for tx in accepted:
            self.state.apply(tx)
        self.chain.append(blk)
        self.mempool = [t for t in self.mempool if t not in accepted]
        return blk

    def is_valid(self) -> Tuple[bool, str]:
        target = "0" * self.difficulty
        for i in range(1, len(self.chain)):
            cur, prev = self.chain[i], self.chain[i-1]
            if cur.compute_hash() != cur.hash:     return False, f"block {i}: hash mismatch"
            if cur.prev_hash != prev.hash:         return False, f"block {i}: broken link"
            if not cur.hash.startswith(target):    return False, f"block {i}: bad pow"
            if cur.timestamp < prev.timestamp:     return False, f"block {i}: non-monotonic ts"
            for j, tx in enumerate(cur.txs):
                if not tx.verify():                return False, f"block {i} tx {j}: bad sig"
        return True, ""

### Demo — end-to-end transaction through a mined block

In [ ]:
alice = Wallet()
bob   = Wallet()

bc = TxBlockchain(difficulty=3, genesis_balances={alice.address(): 100})

# Alice sends 10 to Bob
tx = Transaction(alice.address(), bob.address(), 10, nonce=0)
tx.sign_with(alice)
bc.submit(tx)

# Miner includes the tx in a block
blk = bc.mine_block()
print(f"mined block {blk.index}  txs={len(blk.txs)}  hash={blk.hash[:16]}...")
print("balances:", bc.state.snapshot())
print("chain valid?", bc.is_valid())

### Student Tasks 

#### Tier 1 — Must complete
1. Create three wallets; fund one via `genesis_balances`; mine at least two blocks with cross-sends.
2. Reproduce the three attack demos in your own code, in your own words.
3. Write a test that signs a transaction, flips one bit of `tx.amount`, and confirms `verify()` fails.

#### Tier 2 — Should complete
4. Add **fees**: each transaction has a `fee` field; miners receive the sum of fees as a coinbase transaction in the block.
5. Write a **replay-attack test across chain reorgs**: if the state is rolled back, does your nonce tracking still prevent replay?
6. Add transaction serialisation to JSON and a peer `receive_block(blk_json)` method that validates before accepting.

#### Tier 3 — Stretch
7. Add **deterministic signatures** (RFC 6979) — the `ecdsa` library supports `sign_deterministic`. Discuss why this prevents the Sony PS3-style nonce-reuse attack.
8. Add **UTXO-style transactions** (one output becomes the input of another) instead of the account model. Which model is easier to reason about for security?
9. Expose the whole thing over a tiny Flask / FastAPI interface so two notebooks can talk to each other.


### Suggested in-class live-attack exercise

Volunteer a student's code and introduce **one** of these bugs, then attack it live:

1. Remove the `from_addr == to_addr` check -> Alice "sends" 10 to herself before `from` is debited -> balance doubles.
2. Move the debit *before* the balance check -> state corruption on overdraw.
3. Include `signature` in `signing_hash` -> Alice can *never* produce a valid signature -> silent DoS.
4. Use `random.Random()` (seeded) instead of `secrets` for key generation -> show that predictable RNG produces predictable keys.
5. Forget to re-derive `from_addr` from the public key -> Mallory swaps her own pubkey in, spending Alice's coins.




*This module is deliberately minimal. A real blockchain adds networking, Merkle trees, UTXO vs account models, fork choice, finality, slashing, MEV, and a dozen other concerns. The goal here is that students who can explain why every field on `Block` and `Transaction` exists — and who can break their own code before an attacker does — will pick those up much faster later.*
